## Launch Phoenix

In [3]:
import phoenix as px

px.launch_app()

🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
📖 For more information on how to use Phoenix, check out https://docs.arize.com/phoenix


## Setup Dotenv

In [41]:
import os
import dotenv

dotenv.load_dotenv()

True

## Setup Pandas

In [39]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

## Tracing

### Configure Tracing with Phoenix as destination

In [4]:
from phoenix.otel import register

tracer_provider = register(endpoint="http://127.0.0.1:6006/v1/traces")

🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: default
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://127.0.0.1:6006/v1/traces
|  Transport: HTTP
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



### Instrument Llama Index to generate spans and traces for LLama Index Activity

In [5]:
from openinference.instrumentation.llama_index import LlamaIndexInstrumentor

LlamaIndexInstrumentor().instrument(tracer_provider=tracer_provider, skip_dep_check=True)

Configure default models for Llama Index

In [6]:
from llama_index.core import Settings
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

Settings.llm = OpenAI(model="gpt-4o")
Settings.embed_model = OpenAIEmbedding(model="text-embedding-ada-002")

Configure storage for Llama Index

In [7]:
from gcsfs import GCSFileSystem

fs = GCSFileSystem(project="public-assets-275721")

In [8]:
from llama_index.core import StorageContext

index_path = "arize-phoenix-assets/datasets/unstructured/llm/llama-index/arize-docs/index/"

storage_context = StorageContext.from_defaults(
    fs=fs,
    persist_dir=index_path,
)

Configure LLama Index query engine

In [9]:
from llama_index.core import load_index_from_storage

index = load_index_from_storage(storage_context)
query_engine = index.as_query_engine()

Run some example queries

In [10]:
from tqdm import tqdm

queries = [
    "How can I query for a monitor's status using GraphQL?",
    "How do I delete a model?",
    "How much does an enterprise license of Arize cost?",
    "How do I log a prediction using the python SDK?",
]

for query in tqdm(queries):
    response = query_engine.query(query)
    print(f"{query=}\n{response=}")

 25%|██▌       | 1/4 [00:02<00:07,  2.36s/it]

query="How can I query for a monitor's status using GraphQL?"
response=Response(response='To query for a monitor\'s status using GraphQL, you can use a query structure like the following:\n\n```graphql\nquery {\n  node(id: "monitor_id") {\n    ... on Monitor {\n      status\n    }\n  }\n}\n```\n\nReplace `"monitor_id"` with the actual ID of the monitor you want to query. This will return the status of the specified monitor.', source_nodes=[NodeWithScore(node=TextNode(id_='6360a77d-83d4-4fbd-9a8b-3068fb8ecb76', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='bd3015cc-4467-43a7-a011-2cd8fdf0bad9', node_type=None, metadata={}, hash=None)}, text='\nHere I\'m using a monitor that is triggered from the previous example. You can find a monitor by following the example or by using the UI and grabbing the monitor id from the URL. (`.../monitors/:`**`monitor_id`**).&#x20;\n\n{% ta

 50%|█████     | 2/4 [00:04<00:03,  1.99s/it]

query='How do I delete a model?'
response=Response(response='The provided information does not include instructions on how to delete a model. You may need to refer to the specific documentation or support resources for guidance on deleting a model.', source_nodes=[NodeWithScore(node=TextNode(id_='a5942be3-bf05-482a-bc11-1308e5b19c64', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='ea1c94e9-6177-469f-a167-a735e159d05f', node_type=None, metadata={}, hash=None)}, text="\nNavigate to the custom metric editor in the top navigation bar of any model. Click on 'Custom Metrics' to uncover the editor.&#x20;\n\n", mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_template='{metadata_str}\n\n{content}', metadata_template='{key}: {value}', metadata_seperator='\n'), score=0.7753462502450911), NodeWithScore(node=TextNode(id_='b6b6268c-22ba-4086-8a85-bd06fa658f6c', em

 75%|███████▌  | 3/4 [00:05<00:01,  1.80s/it]

query='How much does an enterprise license of Arize cost?'
response=Response(response='For information regarding the cost of an enterprise license of Arize, it is recommended to contact their sales team directly at contacts@arize.com.', source_nodes=[NodeWithScore(node=TextNode(id_='172fd96f-85e1-4a94-91cb-f417b82b20a9', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={<NodeRelationship.SOURCE: '1'>: RelatedNodeInfo(node_id='fc463e5c-1331-4bf8-9229-175ee502e9ae', node_type=None, metadata={}, hash=None)}, text="\nWe also offer Arize On-Prem, which runs a completely scalable infrastructure within your company's AWS, GCP or Azure account. The environment can be provisioned by us or by your company, using a toolset comprised of Terraform and Kubernetes.\n\nTalk to our sales team for more detailed setup instructions by reaching out to contacts@arize.com\n\n", mimetype='text/plain', start_char_idx=None, end_char_idx=None, text_templa

100%|██████████| 4/4 [00:32<00:00,  8.01s/it]

query='How do I log a prediction using the python SDK?'
response=Response(response="To log a prediction using the Python SDK, you need to use the `arize.log` function. You should provide details such as `prediction_id`, `model_id`, `model_type`, `environment`, `model_version`, `prediction_timestamp`, `features`, `prediction_label`, and any additional `tags`. Here's an example:\n\n```python\nresponse = arize.log(\n    prediction_id='your_prediction_id',\n    model_id='your_model_id',\n    model_type=ModelTypes.SCORE_CATEGORICAL,  # or another appropriate model type\n    environment=Environments.PRODUCTION,  # or another appropriate environment\n    model_version='your_model_version',\n    prediction_timestamp=your_timestamp,\n    features=your_features_dict,\n    prediction_label=('your_label', your_score),\n    tags=your_tags_dict\n)\n```\n\nMake sure to replace placeholders like `'your_prediction_id'`, `'your_model_id'`, and others with your actual data.", source_nodes=[NodeWithScore(

Retrieve the span data from Phoenix

In [11]:
spans_df = px.Client().get_spans_dataframe()
spans_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56 entries, d22bd196aece1dbf to 651970630340cf1f
Data columns (total 27 columns):
 #   Column                                    Non-Null Count  Dtype              
---  ------                                    --------------  -----              
 0   name                                      56 non-null     object             
 1   span_kind                                 56 non-null     object             
 2   parent_id                                 52 non-null     object             
 3   start_time                                56 non-null     datetime64[ns, UTC]
 4   end_time                                  56 non-null     datetime64[ns, UTC]
 5   status_code                               56 non-null     object             
 6   status_message                            56 non-null     object             
 7   events                                    56 non-null     object             
 8   context.span_id                       

/home/lambdakris/source/krs-labs/eval-mon/.venv/lib/python3.11/site-packages/phoenix/trace/dsl/query.py:741: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df_attributes = pd.DataFrame.from_records(


In [12]:
spans_df.head()

,name,span_kind,parent_id,start_time,end_time,status_code,status_message,events,context.span_id,context.trace_id,...,attributes.retrieval.documents,attributes.llm.input_messages,attributes.llm.model_name,attributes.llm.token_count.completion,attributes.llm.token_count.prompt,attributes.llm.token_count.total,attributes.llm.invocation_parameters,attributes.llm.output_messages,attributes.llm.prompt_template.template,attributes.llm.prompt_template.variables
context.span_id,,,,,,,,,,,,,,,,,,,,,
d22bd196aece1dbf,OpenAIEmbedding._get_query_embedding,EMBEDDING,dae15e566ec0989a,2025-01-10 23:43:36.657755+00:00,2025-01-10 23:43:37.062444+00:00,OK,,[],d22bd196aece1dbf,aa581193352a8d4aed7db12823186b65,...,None,None,None,NaN,NaN,NaN,None,None,None,None
dae15e566ec0989a,BaseEmbedding.get_query_embedding,EMBEDDING,5063912e95b2323c,2025-01-10 23:43:36.657101+00:00,2025-01-10 23:43:37.072765+00:00,OK,,[],dae15e566ec0989a,aa581193352a8d4aed7db12823186b65,...,None,None,None,NaN,NaN,NaN,None,None,None,None
5063912e95b2323c,VectorIndexRetriever._retrieve,RETRIEVER,cae5a12e53b7ec13,2025-01-10 23:43:36.656774+00:00,2025-01-10 23:43:37.154918+00:00,OK,,[],5063912e95b2323c,aa581193352a8d4aed7db12823186b65,...,None,None,None,NaN,NaN,NaN,None,None,None,None
cae5a12e53b7ec13,BaseRetriever.retrieve,RETRIEVER,c49c2109d364cc3d,2025-01-10 23:43:36.655743+00:00,2025-01-10 23:43:37.158732+00:00,OK,,[],cae5a12e53b7ec13,aa581193352a8d4aed7db12823186b65,...,[{'document.id': '6360a77d-83d4-4fbd-9a8b-3068...,None,None,NaN,NaN,NaN,None,None,None,None
bfec0fd5545ccc43,TokenTextSplitter.split_text,CHAIN,4b75ad3f68ba1812,2025-01-10 23:43:37.168126+00:00,2025-01-10 23:43:37.168909+00:00,OK,,[],bfec0fd5545ccc43,aa581193352a8d4aed7db12823186b65,...,None,None,None,NaN,NaN,NaN,None,None,None,None


## Evaluation

Derive dataset for question answer pairs from span data

In [13]:
from phoenix.session.evaluation import get_qa_with_reference

qa_dataset = get_qa_with_reference(px.active_session())
qa_dataset.head()

,input,output,reference
context.span_id,,,
19d7edecade7cc2a,How can I query for a monitor's status using G...,"To query for a monitor's status using GraphQL,...",\nHere I'm using a monitor that is triggered f...
9063503fb60a3a54,How do I delete a model?,The provided information does not include inst...,\nNavigate to the custom metric editor in the ...
d045ffaab014c8b4,How much does an enterprise license of Arize c...,For information regarding the cost of an enter...,"\nWe also offer Arize On-Prem, which runs a co..."
651970630340cf1f,How do I log a prediction using the python SDK?,"To log a prediction using the Python SDK, you ...",schema = Schema(\n prediction_id_column_nam...


Derive dataset for retrieved documents from span data

In [14]:
from phoenix.session.evaluation import get_retrieved_documents

doc_dataset = get_retrieved_documents(px.active_session())
doc_dataset.head()

context.trace_id  \
context.span_id  document_position                                     
cae5a12e53b7ec13 0                  aa581193352a8d4aed7db12823186b65   
                 1                  aa581193352a8d4aed7db12823186b65   
855b47e14d50a596 0                  4a94c0a048512297fd7cea1b64cbc8f7   
                 1                  4a94c0a048512297fd7cea1b64cbc8f7   
77b2467e50e54eb7 0                  4d854e0ffc88e81750b23a558e147c04   

                                                                                input  \
context.span_id  document_position                                                      
cae5a12e53b7ec13 0                  How can I query for a monitor's status using G...   
                 1                  How can I query for a monitor's status using G...   
855b47e14d50a596 0                                           How do I delete a model?   
                 1                                           How do I delete a model?   
77b2467e50e54eb7 0                  How much does an enterprise license of Arize c...   

                                                                            reference  \
context.span_id  document_position                                                      
cae5a12e53b7ec13 0                  \nHere I'm using a monitor that is triggered f...   
                 1                  \n```graphql\n{\n  monitor {\n    threshold\n ...   
855b47e14d50a596 0                  \nNavigate to the custom metric editor in the ...   
                 1                  \n> See below for more details, or click to na...   
77b2467e50e54eb7 0                  \nWe also offer Arize On-Prem, which runs a co...   

                                    document_score  
context.span_id  document_position                  
cae5a12e53b7ec13 0                        0.874721  
                 1                        0.852448  
855b47e14d50a596 0                        0.775346  
                 1                        0.774835  
77b2467e50e54eb7 0                        0.822991

With these datasets in place, we can apply evaluations to them to understand how our application is performing.

Instantiate an model for evaluation

In [31]:
from phoenix.evals import OpenAIModel, llm_classify

eval_model = OpenAIModel(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version=os.environ["AZURE_OPENAI_API_VERSION"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
)

evaluate for hallucation

In [32]:
from phoenix.evals import (
    HALLUCINATION_PROMPT_RAILS_MAP,
    HALLUCINATION_PROMPT_TEMPLATE,
)

hallucination_eval = llm_classify(
    dataframe=qa_dataset,
    model=eval_model,
    template=HALLUCINATION_PROMPT_TEMPLATE,
    rails=list(HALLUCINATION_PROMPT_RAILS_MAP.values()),
    provide_explanation=True,
)

🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


llm_classify |          | 0/4 (0.0%) | ⏳ 00:00<? | ?it/s

In [37]:
hallucination_eval[["explanation"]].head()

,explanation
context.span_id,
19d7edecade7cc2a,"To determine if the answer is factual or hallucinated, we first analyze the query, which asks how to query for a monitor's status using GraphQL. The reference text provides a specific GraphQL query structure that retrieves various attributes of a monitor, including its status. The answer provided mirrors this structure, correctly stating that to query for a monitor's status, one can use a query that specifies the node with the monitor's ID and requests the status field. The answer also correctly instructs to replace 'monitor_id' with the actual ID of the monitor. Since the answer aligns with the information in the reference text and accurately reflects the method to query for a monitor's status, it is factual and not a hallucination."
9063503fb60a3a54,"To determine if the answer is factual or hallucinated, we first analyze the query, which asks how to delete a model. Next, we examine the reference text, which provides information about navigating to the custom metric editor and lists various model attributes, but does not mention any instructions or information regarding deleting a model. The answer states that the provided information does not include instructions on how to delete a model and suggests referring to specific documentation or support resources for guidance. This aligns with the reference text, as it confirms the absence of deletion instructions. Therefore, the answer is factual because it accurately reflects the content of the reference text without introducing any false information."
d045ffaab014c8b4,"To determine if the answer is factual or hallucinated, we first analyze the query, which asks about the cost of an enterprise license of Arize. The reference text does not provide any specific information regarding the cost of an enterprise license. Instead, it suggests contacting the sales team for detailed information. The answer correctly reflects this by advising the user to contact the sales team directly for cost information. Since the answer does not introduce any false information and aligns with the reference text's suggestion, it is considered factual."
651970630340cf1f,"To determine if the answer is factual or hallucinated, we first analyze the query, which asks how to log a prediction using the Python SDK. The reference text provides specific details on how to use the `arize.log` function, including the parameters that need to be provided such as `prediction_id`, `model_id`, `model_type`, `environment`, `model_version`, `prediction_timestamp`, `features`, `prediction_label`, and `tags`. The answer correctly outlines these parameters and provides a code example that aligns with the information in the reference text. It also mentions the need to replace placeholders with actual data, which is consistent with the guidance in the reference text. Since the answer accurately reflects the content and instructions from the reference text without introducing any false information, it is deemed factual."


evaluate for correctness

In [34]:
from phoenix.evals import (
    QA_PROMPT_RAILS_MAP, 
    QA_PROMPT_TEMPLATE,
)

qa_eval = llm_classify(
    dataframe=qa_dataset,
    model=eval_model,
    template=QA_PROMPT_TEMPLATE,
    rails=list(QA_PROMPT_RAILS_MAP.values()),
    provide_explanation=True,
    concurrency=4,
)

🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


llm_classify |          | 0/4 (0.0%) | ⏳ 00:00<? | ?it/s

In [35]:
qa_eval.head()

,label,explanation,exceptions,execution_status,execution_seconds
context.span_id,,,,,
19d7edecade7cc2a,correct,The answer provides a valid GraphQL query stru...,[],COMPLETED,1.441879
9063503fb60a3a54,correct,The answer states that the provided informatio...,[],COMPLETED,4.606905
d045ffaab014c8b4,incorrect,The answer states that for information regardi...,[],COMPLETED,2.968731
651970630340cf1f,correct,The answer provides a detailed explanation of ...,[],COMPLETED,1.267146


Upload evaluation results to Phoenix

In [40]:
from phoenix.trace import SpanEvaluations

px.Client().log_evaluations(
    SpanEvaluations(eval_name="Hallucination", dataframe=hallucination_eval),
    SpanEvaluations(eval_name="QA", dataframe=qa_eval),
)